# 02 — Transformação e Limpeza dos Dados

Pipeline: filtrar Influenza, tipar colunas, calcular faixa etária, agregar por UF/mês.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import polars as pl
import pyarrow.dataset as pds
import pyarrow.fs as fs
import pyarrow.compute as pc

from src.config import S3_ENDPOINT, S3_ACCESS_KEY, S3_SECRET_KEY, S3_REGION, S3_BUCKET, ANOS, COLUNAS_PNI
from src.transform import (
    filtra_influenza_pni, filtra_sih_sim_influenza,
    tipa_pni, faixa_etaria, adiciona_regiao, adiciona_mes_ano,
    agrega_doses, prepara_pni_completo,
)

In [ ]:
# Carregar dados de uma UF (ex: AC - Acre)
from src.ingest import load_pni_uf

table = load_pni_uf('AC', anos=['2025'], meses=['05','06','07'])
print(f"Total registros PNI: {table.num_rows:,}")

In [ ]:
# Filtrar apenas Influenza
from src.transform import filtra_influenza_pni

table_influenza = filtra_influenza_pni(table)
print(f"Registros Influenza: {table_influenza.num_rows:,}")

In [ ]:
# Aplicar transformações
df = prepara_pni_completo(table_influenza)
print(f"Shape: {df.shape}")
print(f"Colunas: {list(df.columns)}")
df.head()

In [ ]:
# Agregar por UF + mês
doses_agg = agrega_doses(df, "uf_mes").to_pandas()
doses_agg.head(10)

In [ ]:
# Carregar SIH e filtrar Influenza
from pysus.online_data.SIH import download

df_sih = download("SP", 2025)
df_sih_influenza = filtra_sih_sim_influenza(df_sih, "DIAG_PRINC")
print(f"Internações Influenza SP/2025: {len(df_sih_influenza)}")
df_sih_influenza.head()

In [ ]:
# Salvar dados processados
df.to_parquet("../data/processed/pni_influenza.parquet", index=False)
doses_agg.to_csv("../data/processed/doses_agg_uf_mes.csv", index=False)
print("Dados salvos em data/processed/")

---
**Próximo notebook**: `03_analysis.ipynb` — análise completa